In [55]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

In [56]:
# Rutas
RUTA_LECTURA = r"C:\Users\ramir\OneDrive\Escritorio\Simulador\Repositorio\ProjecteData\Equip_30\Data\06-01-2026\06-01-2026_Clean.csv"
RUTA_ESCRITURA = r"C:\Users\ramir\OneDrive\Escritorio\Simulador\Repositorio\ProjecteData\Equip_30\Data\06-01-2026\06-01-2026_Marketing_Analysis.csv"

In [57]:
# Cargar datos 
df_marketing = pd.read_csv(RUTA_LECTURA)

In [58]:
df_marketing["perfil_deuda"] = np.where(
    (df_marketing["housing"] == 1) | (df_marketing["loan"] == 1), 'With Debt', 'No Debt'
)

In [59]:
display(df_marketing)

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,perfil_deuda
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_pcampaign,1,With Debt
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_pcampaign,1,No Debt
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_pcampaign,1,With Debt
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_pcampaign,1,With Debt
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_pcampaign,1,No Debt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10982,10983,40,management,married,secondary,0,8486,0,0,unknown,6,may,260,3,-1,0,no_pcampaign,0,No Debt
10983,10984,53,management,married,tertiary,0,20772,0,0,cellular,4,feb,715,1,-1,0,no_pcampaign,0,No Debt
10984,10985,55,blue-collar,married,primary,0,3297,1,1,telephone,30,apr,96,1,-1,0,no_pcampaign,0,With Debt
10985,10986,41,management,married,tertiary,0,9,1,0,cellular,22,jul,82,3,-1,0,no_pcampaign,0,With Debt


In [60]:
# Revisar el cruce de las variables principales para asegurar volumen de datos
tabla_cruce = pd.crosstab(df_marketing['contact'], df_marketing['deposit'])
print("--- Validación de Tamaño de Muestra ---")
print(tabla_cruce)


--- Validación de Tamaño de Muestra ---
deposit       0     1
contact              
cellular   3559  4369
telephone   374   390
unknown    1765   530


In [61]:
# Variable del Perfil de Deuda (Base de referencia será: 'With Debt')
df_marketing['debt_profile'] = np.where((df_marketing['housing'] == 1) | (df_marketing['loan'] == 1), 'With Debt', 'No Debt')

# Tramos de Balance (Base de referencia será: 'Negative or Zero')
condiciones_balance = [
    (df_marketing['balance'] <= 0),
    (df_marketing['balance'] > 0) & (df_marketing['balance'] <= 556),
    (df_marketing['balance'] > 556) & (df_marketing['balance'] <= 2000),
    (df_marketing['balance'] > 2000)
]
opciones_balance = ['Negative or Zero', 'Low Balance', 'Medium Balance', 'High Balance']
df_marketing['balance_tier'] = np.select(condiciones_balance, opciones_balance, default='Negative or Zero')

# Categorizar la Edad en Tramos Generacionales para el Negocio (Base de referencia será: 'Adult (36-55)')
condiciones_edad = [
    (df_marketing['age'] <= 35),
    (df_marketing['age'] > 35) & (df_marketing['age'] <= 55),
    (df_marketing['age'] > 55)
]
opciones_edad = ['Young (<=35)', 'Adult (36-55)', 'Senior (>55)']
df_marketing['age_tier'] = np.select(condiciones_edad, opciones_edad, default='Adult (36-55)')

# MODELO
# Forzamos a Python a usar 'telephone', 'With Debt', 'Negative or Zero' 'Adult (36-55)' como bases
formula = """
deposit ~ C(contact, Treatment(reference='telephone')) * C(age_tier, Treatment(reference='Adult (36-55)')) 
         + C(contact, Treatment(reference='telephone')) * C(debt_profile, Treatment(reference='With Debt')) 
         + C(balance_tier, Treatment(reference='Negative or Zero')) 
         + C(poutcome) 
         + campaign
"""

modelo_profesional = smf.logit(formula, data=df_marketing).fit()

# 6. Imprimir los resultados limpios (Odds Ratios)
resultados_profesionales = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_profesional.params),
    'P-valor': modelo_profesional.pvalues
})

print(f"\nPSEUDO R-SQUARED DE MCFADDEN FINAL: {modelo_profesional.prsquared:.4f}")
print("\n--- RESULTADOS FINALES PROFESIONALES RE-ESTRUCTURADOS ---")
print(resultados_profesionales.round(4))


Optimization terminated successfully.
         Current function value: 0.592763
         Iterations 6

PSEUDO R-SQUARED DE MCFADDEN FINAL: 0.1440

--- RESULTADOS FINALES PROFESIONALES RE-ESTRUCTURADOS ---
                                                    Odds Ratio (OR)  P-valor
Intercept                                                    0.4081   0.0000
C(contact, Treatment(reference='telephone'))[T....           1.4679   0.0163
C(contact, Treatment(reference='telephone'))[T....           0.7030   0.0414
C(age_tier, Treatment(reference='Adult (36-55)'...           2.1037   0.0001
C(age_tier, Treatment(reference='Adult (36-55)'...           1.1204   0.5866
C(debt_profile, Treatment(reference='With Debt'...           2.2446   0.0000
C(balance_tier, Treatment(reference='Negative o...           1.7910   0.0000
C(balance_tier, Treatment(reference='Negative o...           1.1472   0.0462
C(balance_tier, Treatment(reference='Negative o...           1.4869   0.0000
C(poutcome)[T.no_pcampaig

In [62]:
# Ver el resumen estadístico completo
modelo_profesional = smf.logit(formula, data=df_marketing).fit()
print(modelo_profesional.summary())

Optimization terminated successfully.
         Current function value: 0.592763
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                10987
Model:                          Logit   Df Residuals:                    10968
Method:                           MLE   Df Model:                           18
Date:                Sun, 07 Jun 2026   Pseudo R-squ.:                  0.1440
Time:                        17:41:42   Log-Likelihood:                -6512.7
converged:                       True   LL-Null:                       -7608.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                                                                                 coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

In [63]:
display(resultados_profesionales.round(4))

,Odds Ratio (OR),P-valor
Intercept,0.4081,0.0000
"C(contact, Treatment(reference='telephone'))[T.cellular]",1.4679,0.0163
"C(contact, Treatment(reference='telephone'))[T.unknown]",0.7030,0.0414
"C(age_tier, Treatment(reference='Adult (36-55)'))[T.Senior (>55)]",2.1037,0.0001
"C(age_tier, Treatment(reference='Adult (36-55)'))[T.Young (<=35)]",1.1204,0.5866
"C(debt_profile, Treatment(reference='With Debt'))[T.No Debt]",2.2446,0.0000
"C(balance_tier, Treatment(reference='Negative or Zero'))[T.High Balance]",1.7910,0.0000
"C(balance_tier, Treatment(reference='Negative or Zero'))[T.Low Balance]",1.1472,0.0462
"C(balance_tier, Treatment(reference='Negative or Zero'))[T.Medium Balance]",1.4869,0.0000
C(poutcome)[T.no_pcampaign],0.8611,0.0237


In [64]:
df_resultados = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_profesional.params),
    'P-valor': modelo_profesional.pvalues
}).round(4)
print("\n--- RESULTADOS FINALES DEL MODELO ---")
print(df_resultados)


--- RESULTADOS FINALES DEL MODELO ---
                                                    Odds Ratio (OR)  P-valor
Intercept                                                    0.4081   0.0000
C(contact, Treatment(reference='telephone'))[T....           1.4679   0.0163
C(contact, Treatment(reference='telephone'))[T....           0.7030   0.0414
C(age_tier, Treatment(reference='Adult (36-55)'...           2.1037   0.0001
C(age_tier, Treatment(reference='Adult (36-55)'...           1.1204   0.5866
C(debt_profile, Treatment(reference='With Debt'...           2.2446   0.0000
C(balance_tier, Treatment(reference='Negative o...           1.7910   0.0000
C(balance_tier, Treatment(reference='Negative o...           1.1472   0.0462
C(balance_tier, Treatment(reference='Negative o...           1.4869   0.0000
C(poutcome)[T.no_pcampaign]                                  0.8611   0.0237
C(poutcome)[T.other]                                         1.2710   0.0278
C(poutcome)[T.success]               

In [65]:
df_resultados.to_csv(RUTA_ESCRITURA, index=True, sep=',', encoding='utf-8')

print(f"Archivo guardado exitosamente en: {RUTA_ESCRITURA}")

Archivo guardado exitosamente en: C:\Users\ramir\OneDrive\Escritorio\Simulador\Repositorio\ProjecteData\Equip_30\Data\06-01-2026\06-01-2026_Marketing_Analysis.csv
